# Using Memory-Based State Estimators

In [15]:
#| hide
from nbdev import show_doc

# Introduction

In this tutorial, we show how to use a series of increasingly complex memory graphs as the basis for Bayesian updates to a belief about the state of an external environment.

In the simplest memory graph,  'Sequence Memory', we model the memorization of an exact sequence of percepts through the encoding of percepts into memory traces (the graph nodes). We will show how this memory graph can be used to establish the probability that an external environment has returned to the exact state or sequence of states that produced the memorized sequence.

We add a level of complexity in 'Short-Term Memory', which models the 'fading' of information encoded in memory traces. We will show how this fading is comparable to forming a coarse-grained model of the world, making state estimation on the memory graph more robust to noisy signals.

'Long-Term Memory' builds further complexity by modeling (1) how the surprise of each percept mediates the fading rate of the memory trace to which it is encoded and (2) how 're-activation' of a memory trace consolidates the memory by slowing the fading rate of that trace. We will show how this allows the memory graph to prioritize the retention and re-activation of relevant memory traces, setting the stage for learning efficient representations of the external environment

Finally, with 'Associative Memory', we model the plasticity of memory traces upon re-activation. We will show how this plasticity allows the memory graph to learn and use efficient represetations of the external environnment for estimating the current state of that environment.

# Background - Series Recognition

Throughout this tutorial, we will use an experimental paradigm from psychology called "Series Recognition" to illustrate the functionality of each memory graph and the aspects of memory which it is said to model. In a Series Recognition task, a human or animal subject is presented with a series of items (i.e. a list). After a period of time known as a 'retention interval', the subject is presented with another item ('probe') and asked to identify whether or not that item was in the recently preseneted list. In cases where the probe matches an item from the recently presented list ('target'), the probability that both humans and animals answer correctly depends both on the position of the target in the list (e.g. first, second, fifth, last, etc.) and the length of the retention interval. Subjects are more likely to accurately recognize items presented earlier ('primacy effect') and items presented toward the end of the list ('recency effect') than items in the middle. For short retention intervals, the recency effect is much stronger than the primacy effect; as the retention interval increases, the recency effect reduces to the point of vanishing while the primacy effect remains consistent or increases. 

This interaction is widely regarded as evidence of a process of memory traces (sometimes 'stimulus traces') in short-term memory either being forgotten or transferred to long-term memory. We will use simulated environments that emulate a Series Recognition task to illustrate how each kind of memory graph builds toward a system of memory that both reproduces these effects and also learns efficient representations of the environment that can be used to make predictions about the future.

# Sequence Memory

With the "Sequence Memory" memory graph, we introduce the key concepts of memory-based state estimation. These key concepts are (1) The Memory-Environment Interface, (2) The Stationary Hypothesis, (3) Memory Traces, and (4) Bayesian Filtering. We will show that with these concepts, Sequence Memory can be usefully applied for state estimation in certain contrived environments.

## The Memory-Environment Interface

In the general Projective Simulation framework, an agent is said to interact with its environment through a percept-action loop. In this loop, the environment has a state which is updated as a function of the agent's action. This new environment state produces a percept, and the agent then updates its own state as a function of this percept before producing a new action. Because we are particularly interested in the *embodied* nature of agency, we think of that part of the agent which 'receives' the percept and 'transmits' the action as the agent-environment interface.

In this work, we consider a memory graph as a component of such an agent - a component that may form the basis of an Episodic and Compositional Memory. This memory graph takes as input the same percept as the agent and updates its state likewise. Because we do not explicitly model the production of an action by the agent here, we consider the memory-environment interface to be indentical the agent-environment *sensory* interface. An important aspect of our memory graphs is that they decompose a percept into consituent parts - one can think of each percept 'category' being converted to the state of a corresponding sensor in the memory-environment interface.

To define a memory-graph, then, the first thing one must define is the number of sensors in the memory-environment interface and the number of states each sensor can take. This information is passed to the Sequence_Memory constructor as the argument 'category_sizes'. We will cover the other arguments to Sequence_Memory next.

In [16]:
from projective_simulation.ECMs.state_estimators import Sequence_Memory

In [17]:
show_doc(Sequence_Memory)

---

[source](https://github.com/{user}/projective_simulation/blob/master/projective_simulation/ECMs/state_estimators.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### Sequence_Memory

>      Sequence_Memory (category_sizes:list[int], memory_capacity:int,
>                       memory_bias:float, stationary_expectations:numpy.ndarray
>                       [typing.Any,numpy.dtype[numpy.float64]]=None, sensory_pr
>                       edictions:numpy.ndarray[typing.Any,numpy.dtype[numpy.flo
>                       at64]]=None, belief_prior:numpy.ndarray[typing.Any,numpy
>                       .dtype[numpy.float64]]=None, transition_predictions:nump
>                       y.ndarray[typing.Any,numpy.dtype[numpy.float64]]=None,
>                       timer:int=0, capacity_overflow_method:str='stop
>                       encoding', data_record:list[str]=[],
>                       record_until:int=-1)

*Memory-based Bayesian filter that encodes a sequence of percepts as memory traces.
Each memory trace is a hypothesis about the environment, and the agent can transition between non-memory and memory hypotheses. 
Memory traces are encoded by dynamic modification of transition function and observation function.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| category_sizes | list |  | Number of states for each sensor/percept category. |
| memory_capacity | int |  | Number of memory-based hypotheses. |
| memory_bias | float |  | Probability of transitioning from non-memory to memory hypothesis space. |
| stationary_expectations | ndarray | None | Expectation of sensory states in stationarity. Default to uniform distributions for each sensor. |
| sensory_predictions | ndarray | None | Sensory hypotheses matrix. If None, initialized to uniform. |
| belief_prior | ndarray | None | Initial belief priors. If None, all prior on non-memory hypothesis. |
| transition_predictions | ndarray | None | Hypothesis transition matrix. |
| timer | int | 0 | Starting memory time index. |
| capacity_overflow_method | str | stop encoding | 'loop' or 'stop encoding'. |
| data_record | list | [] | List of variable names to log each time step. Accepts "all". |
| record_until | int | -1 | Number of steps to prepare for data logging. Negative disables recording. |

## The Stationary Hypothesis

The goal of a memory graph is to build a predictive model of the environment for an agent that has no prior knowledge about the structure of the environment, i.e. the states of the environment that exist and how they relate to each other and to the agent's percepts. To achieve this, it is helpful to assume the agent 'inherits' some knowledge about general properties of its environment; what we might call *implicit biases*. In our case, we assume that the agent inherits knowledge of the overall probability that any given state in the environment yields a given state for any sensor - or at least a rough approximation thereof. This knowledge is represented in a memory graph by what we call the "stationary hypothesis". To define the stationary hypothesis, one must pass a 1D numpy array to the argument stationary_expectations of the Sequence_Memory constructor that contains these probabilities. These probabilites must be ordered such that the first $N_1$ elements represent the probability distribution over the $N_1$ possible states of the first sensor, as defined by category_sizes; the next $N_2$ elements give the probability distribution over the possible states of the second sensor, and so-on and so-forth for all sensors indicated by the length of the category_sizes list. An example is provided below.

In [18]:
import numpy as np

In [ ]:
#Define a memory-environment interface with two sensors, each of which can take two states
category_sizes = [2,2]

#Define stationary_expectations for the above memory-environment interface.
#The resulting implicit bias is that both states of the first sensor are equally common, 
#but the first state of the second sensor occurs 9x more frequently, overall, than the second state.
stationary_expectations_good = np.array([0.5,0.5,0.9,0.1])

#Some examples of invalid stationary_expectations. 
#These would raise Value Errors if initialization of Sequence_Memory was attempted
stationary_expectations_bad1 = np.array([0.5,0.5,0.8,0.1]) #expectations on the second sensor do not yield a valid probability distribution
stationary_expectations_bad2 = np.array([0.5,0.5,0.6,0.3,0.1]) #the total number of possible states does not align with category_sizes

In [20]:
#Initialize Valid Sequence Memory
toy_sequence_memory_graph = Sequence_Memory(
    category_sizes=category_sizes,
    stationary_expectations=stationary_expectations_good,
    memory_capacity=5, #to be explained next
    memory_bias = 0.05 #to be explained subsequently
)
toy_sequence_memory_graph

## Memory Traces

### Encoding Percepts and Sensory Predictions

To begin, the only information a memory graph has about the environment comes from its stationary hypothesis - but it can acquire more information by storing information about its percepts into a **memory trace**. The Sequence Memory graph will store the first $C$ many percepts it receives to memory traces, where $C$ is its *memory capacity*. The percept is initially stored as a set of **one-hot encodings** in the same structure as the expectations encoded by the stationary hypothesis. That is to say, as a 1D array where the first $N_1$ elements are a one-hot encoding of the state of the first percept category/sensor, so-on and so-forth for all percept categories/sensors. 

The combination of the memory graph's memory traces and its stationary hypothesis form the memory graph's **hypothesis space** about its *situation* in the environment. We use *situation* to mean the set of world states (the joint state of the environment and the agent or memory graph) that yield the same effective future for the agent or memory graph's interface with the environment, i.e. the future percepts of the memory graph. For a Sequence Memory graph, one can think of a memory trace as representing the hypothesis that "my situation is the same as when the percept stored in this trace was encoded." The Stationary Hypothesis can be understood to represent the hypothesis "my situation is not the same as it was when the percepts stored in any of my memory traces were encoded." 

The percept information encoded in each of these hypotheses, then, can be understood as a prediction about the state of each sensor/percept category given that the hypothesis is true. In a Sequence_Memory instance, these predictions are stored in the *sensory_predictions* matrix. We can see in our toy instantiation that this matrix is initialized with predictions for each of the five memory traces it has the capacity to encode that are identical to the predictions of its stationary hypothesis.

In [ ]:
toy_sequence_memory_graph.sensory_predictions
#The first five rows of this matrix give the sensory predictions of the five memory traces available to the memory graph.
#The last row gives the sensory predictions of the stationary hypothesis.
#Initially, these are all identical.

array([[0.5, 0.5, 0.9, 0.1],
       [0.5, 0.5, 0.9, 0.1],
       [0.5, 0.5, 0.9, 0.1],
       [0.5, 0.5, 0.9, 0.1],
       [0.5, 0.5, 0.9, 0.1],
       [0.5, 0.5, 0.9, 0.1]])

If we pass a percept to the Sequence Memory graph using its *sample* method, we can see how the first memory trace changes to a one-hot encoding of the percept.

In [23]:
percept1 = np.array([1,0])
toy_sequence_memory_graph.sample(percept1)
toy_sequence_memory_graph.sensory_predictions

array([[0. , 1. , 1. , 0. ],
       [0.5, 0.5, 0.9, 0.1],
       [0.5, 0.5, 0.9, 0.1],
       [0.5, 0.5, 0.9, 0.1],
       [0.5, 0.5, 0.9, 0.1],
       [0.5, 0.5, 0.9, 0.1]])

The default Sequence Memory stops encoding percepts to memory traces after it has used all available traces. We discuss alternative methods in the "further modelling considerations" section.

### Encoding Hypothesis Transitions and Memory Bias

Each of the memory graph's hypotheses store information about sensor state probabilities, give the hypothesis is true, and about the proability that any other hypothesis will be true when the *next* percept is recieved. For a Sequence Memory Graph, this infomormation can be specified on initialization and does not change. The predicted probability of transitioning from one memory trace hypothesis to another is always one if the second memory trace was encoded immediately after the first and zero otherwise. This reflects another implicit bias of the Sequence Memory graph: the assumption that situations follow a deterministic cycle (this assumption will be relaxed in other memory graphs). We can see that these predictions are already set up in the toy Sequence Memory's *transition_predictions*.

In [24]:
toy_sequence_memory_graph.transition_predictions

array([[0.  , 1.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 1.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 1.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 1.  ],
       [0.05, 0.  , 0.  , 0.  , 0.  , 0.95]])

What about the transition predictions of the stationary hypothesis, given by the last row of the matrix? These are defined by the *memory bias*, the last required argument for initializing a Sequence Memory instance. One can see that for a Sequence Memory graph, the memory bias gives the prbability with which the stationary hypothesis predicts the memory graph's next situation will be the same as it's situation when its first memory was encoded. This probability reflects an implicit bias of the Sequence Memory graph regarding the assumed size of the environment's state space. In the case of our toy instance, for example, we can determine that the memory bias implies an expectation that the memory graph will move through a deterministic cycle of twenty-five situations. Why? The memory graph can encode five situations to memory, then must rely on its stationary hypothesis to represent the rest. If there are twenty such remaining situations, and all the memory graph 'knows' if its stationary hypothesis is true is that it is in one of those situations, then the probablility that it is in the situation that precedes the situation represented by its first memory is 0.05.